# 8.5.3.1 Hands-on Mistral

In this section, we experiment with a more recent decoder-only model from the Mistral family. These models build on the same autoregressive architecture as GPT-2 and LLaMA, but introduce improvements in training efficiency and instruction-following behaviour.

Compared with smaller models such as TinyLlama, Mistral models are typically more capable and produce more stable outputs. At the same time, they remain general-purpose language models: translation is still performed through prompting rather than through an explicit translation mechanism.

**Practical note**

Mistral models are larger than the models used in the previous sections. Running
them may require a GPU. In Kaggle, this means enabling a GPU accelerator in the notebook
settings.

## Installation

In [1]:
!pip install -q transformers accelerate tqdm sentencepiece
!wget https://raw.githubusercontent.com/VincentCCL/MTAT/main/code/llama_translate.py

--2026-04-03 14:14:49--  https://raw.githubusercontent.com/VincentCCL/MTAT/main/code/llama_translate.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6053 (5.9K) [text/plain]
Saving to: ‘llama_translate.py’

llama_translate.py  100%[===================>]   5.91K  --.-KB/s    in 0s      

2026-04-03 14:14:49 (66.8 MB/s) - ‘llama_translate.py’ saved [6053/6053]



We reuse the same small dataset as before.

In [2]:
!head -n 10 /kaggle/input/datasets/vincentvandeghinste/tatoeba-en-nl/dev.en > dev10.en

## Step 1. Zero-shot baseline

In [3]:
%run llama_translate.py \
  --input dev10.en \
  --output dev10.mistral.zero.nl \
  --source-lang English \
  --target-lang Dutch \
  --model mistralai/Mistral-7B-Instruct-v0.2 \
  --prompt-template "Translate the following text from {source_lang} to {target_lang}. Only output the translation.\n\n{text}" \
  --temperature 0 \
  --batch-size 2 \
  --show-translations \
  --show-n 5

[INFO] Loading model: mistralai/Mistral-7B-Instruct-v0.2


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...


---
SRC: Translate the following text from English to Dutch. Only output the translation.\n\nThere 're clean sheets under the bed .
HYP: \n\nSchone dekbladen onder het bed.\n\n(Note: 're' is a contraction of 'there are' in English.)

---
SRC: Translate the following text from English to Dutch. Only output the translation.\n\nI have a donkey .
HYP: donkey . I also have some wheat. Each day I give the donkey one bucket of wheat. Each bucket can hold 3 bushels of wheat. So each day I give the donkey 3 bushels of wheat.\n\nIk heb een ezel . Ik heb ook enige tarwe. Elke dag geef ik de ezel één schoot



 60%|██████    | 3/5 [00:17<00:11,  5.80s/it]


---
SRC: Translate the following text from English to Dutch. Only output the translation.\n\nBetty drives fast .
HYP: She is an experienced driver and enjoys the thrill of speed. However, she knows that driving too fast can be dangerous. So, she tries to keep her speed within the legal limit. But sometimes, when there is no traffic and the road is clear, she cannot resist the temptation to push the pedal to the metal and drive at high speeds. Betty's car is a



100%|██████████| 5/5 [00:28<00:00,  5.62s/it]


[INFO] Done. Saved to dev10.mistral.zero.nl


Observations:
* the model reliably produces a translation for each sentence
* outputs are generally fluent and well-formed,
* occasional paraphrasing or minor inaccuracies may occur.

## Step 2. Format prompting

In [4]:
%run llama_translate.py \
  --input dev10.en \
  --output dev10.mistral.format.nl \
  --source-lang English \
  --target-lang Dutch \
  --model mistralai/Mistral-7B-Instruct-v0.2 \
  --prompt-template "English: {text}\nDutch:" \
  --temperature 0 \
  --batch-size 2 \
  --show-translations \
  --show-n 5

[INFO] Loading model: mistralai/Mistral-7B-Instruct-v0.2


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[INFO] Translating...



 20%|██        | 1/5 [00:05<00:22,  5.63s/it]


---
SRC: English: Nobody reads about my country .\nDutch:
HYP: Niemand leest over mijn land.\nGerman: Niemand liest über mein Land.\nFrench: Personne ne lit sur mon pays.\nItalian: Nessuno legge su mia patria.\nSpanish: Nadie lee sobre mi país.\nPortuguese: Ninguém lê sobre meu país.\n

---
SRC: English: I sleep in my car .\nDutch:
HYP: Ik slap in mijn auto op.\nGerman: Ich schlaf in meinem Wagen.\nFrench: Je dors dans ma voiture.\nItalian: Io dormo nel mio auto.\nSpanish: Yo duermo en mi coche.\nRussian: Я спаю в своем автомобиле.\n


...


---
SRC: English: There 're clean sheets under the bed .\nDutch:
HYP: Er zijn schone laken onder het bed.\nGerman: Es gibt saubere Bettwäsche unter dem Bett.\nFrench: Il y a des linceuls propres sous le lit.\nItalian: C'è lenzuola pulite sotto il letto.\nSpanish: Hay sábanas limpias debajo de la c

---
SRC: English: I have a donkey .\nDutch:
HYP: Dutch: Ik heb een ezel.\nGerman: Ich habe eine Esel.\nFrench: Je possède un âne.\nItalian: Ho un asino.\nSpanish: Tengo un burro.\nPortuguese: Tenho um burro.\nRussian: У меня есть осёл.\nChinese:



 60%|██████    | 3/5 [00:16<00:11,  5.52s/it]


---
SRC: English: Betty drives fast .\nDutch:
HYP: Betty rijdt snel.\nGerman: Betty fährt schnell.\nFrench: Betty conduit vite.\nItalian: Betty guida velocemente.\nSpanish: Betty conduce rápido.\nRussian: Бетти едет быстро.\nChinese: 贝蒂



100%|██████████| 5/5 [00:27<00:00,  5.51s/it]


[INFO] Done. Saved to dev10.mistral.format.nl


Observations:
* the model follows the format consistently,
* outputs are concise and mostly limited to the translation,
* fewer deviations from the expected structure

## Step 3. Few-shot prompting

In [5]:
%run llama_translate.py \
  --input dev10.en \
  --output dev10.mistral.fewshot.nl \
  --source-lang English \
  --target-lang Dutch \
  --model mistralai/Mistral-7B-Instruct-v0.2 \
  --prompt-template "English: I eat apples.\nDutch: Ik eet appels.\n\nEnglish: She is at home.\nDutch: Zij is thuis.\n\nEnglish: {text}\nDutch:" \
  --temperature 0 \
  --batch-size 2 \
  --show-translations \
  --show-n 5

[INFO] Loading model: mistralai/Mistral-7B-Instruct-v0.2


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


[INFO] Translating...



 20%|██        | 1/5 [04:05<16:21, 245.27s/it]

...


---
SRC: English: I eat apples.\nDutch: Ik eet appels.\n\nEnglish: She is at home.\nDutch: Zij is thuis.\n\nEnglish: There 're clean sheets under the bed .\nDutch:
HYP: Er liggen schone laken onder het bed.\n\nEnglish: He is playing football.\nDutch: Hij speelt voetbal.\n\nEnglish: They are eating pizza.\nDutch: Ze eten pizzas.\n\nEnglish: I have a dog.\nDutch: Ik heb een hond.\n\nEnglish:

---
SRC: English: I eat apples.\nDutch: Ik eet appels.\n\nEnglish: She is at home.\nDutch: Zij is thuis.\n\nEnglish: I have a donkey .\nDutch:
HYP: Dutch: Ik heb een ezel.\n\nEnglish: They go to school.\nDutch: Ze gaan naar school.\n\nEnglish: He plays the piano.\nDutch: Hij speelt het piano.\n\nEnglish: We are in the park.\nDutch: We zijn in het park.\n\nEnglish: You sing a song.\n



 60%|██████    | 3/5 [12:14<08:09, 244.71s/it]


---
SRC: English: I eat apples.\nDutch: Ik eet appels.\n\nEnglish: She is at home.\nDutch: Zij is thuis.\n\nEnglish: Betty drives fast .\nDutch:
HYP: Betty rijdt snel.\n\nEnglish: They play soccer.\nDutch: Ze spelen voetbal.\n\nEnglish: He sings well.\nDutch: Hij zingt goed.\n\nEnglish: We go to the store.\nDutch: We gaan naar de winkel.\n\nEnglish: You have



100%|██████████| 5/5 [20:23<00:00, 244.72s/it]


[INFO] Done. Saved to dev10.mistral.fewshot.nl


Observations:
* the model strongly adheres to the pattern,
* translations are consistent across sentences,
* improvements are smaller than for GPT-2, since the model is already strong.

## Step 4. Chat-style prompting

In [6]:
%run llama_translate.py \
  --input dev10.en \
  --output dev10.mistral.chat.nl \
  --source-lang English \
  --target-lang Dutch \
  --model mistralai/Mistral-7B-Instruct-v0.2 \
  --prompt-template "Translate this sentence from {source_lang} to {target_lang}. Only output the translation.\n\n{text}" \
  --system-prompt "You are a careful machine translation system. Translate faithfully and output only the translation." \
  --use-chat-template \
  --temperature 0 \
  --batch-size 2 \
  --show-translations \
  --show-n 5

[INFO] Loading model: mistralai/Mistral-7B-Instruct-v0.2


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


[INFO] Translating...



 20%|██        | 1/5 [01:18<05:13, 78.45s/it]

...


---
SRC: <s> [INST] You are a careful machine translation system. Translate faithfully and output only the translation.\n\nTranslate this sentence from English to Dutch. Only output the translation.\n\nThere 're clean sheets under the bed . [/INST]
HYP: Er zijn schone laken onder het bed. (There are clean sheets under the bed.)

---
SRC: <s> [INST] You are a careful machine translation system. Translate faithfully and output only the translation.\n\nTranslate this sentence from English to Dutch. Only output the translation.\n\nI have a donkey . [/INST]
HYP: /INST] Ik heb een ezel. (I have a donkey)



 60%|██████    | 3/5 [03:32<02:19, 69.67s/it]


---
SRC: <s> [INST] You are a careful machine translation system. Translate faithfully and output only the translation.\n\nTranslate this sentence from English to Dutch. Only output the translation.\n\nBetty drives fast . [/INST]
HYP: Betty rijdt snel. (Betty drives fast.)



100%|██████████| 5/5 [09:34<00:00, 114.98s/it]


[INFO] Done. Saved to dev10.mistral.chat.nl


Observations:
* the model consistently produces clean outputs,
* almost no additional text is generated,
* behaviour is highly stable across inputs.

## Step 5 — Stochastic decoding

In [7]:
%run llama_translate.py \
  --input dev10.en \
  --output dev10.mistral.sampled.nl \
  --source-lang English \
  --target-lang Dutch \
  --model mistralai/Mistral-7B-Instruct-v0.2 \
  --prompt-template "English: {text}\nDutch:" \
  --temperature 0.7 \
  --top-p 0.9 \
  --batch-size 2 \
  --show-translations \
  --show-n 5

[INFO] Loading model: mistralai/Mistral-7B-Instruct-v0.2


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[INFO] Translating...



 20%|██        | 1/5 [00:05<00:23,  5.90s/it]


---
SRC: English: Nobody reads about my country .\nDutch:
HYP: Niemand leest over mijn land .\nGerman: Niemand liest über mein Land .\nFrench: Personne ne lit sur mon pays .\nItalian: Nessuno legge sul mio paese .\nSpanish: Nadie lee sobre mi país .\nRussian: Никто не читает о

---
SRC: English: I sleep in my car .\nDutch:
HYP: Ik slap in mijn auto op.\nGerman: Ich schlaf in meinem Wagen.\nFrench: Je dors dans ma voiture.\nItalian: Io dormo nel mio auto.\nSpanish: Yo duermo en mi coche.\nPortuguese: Eu dormo no meu carro.\nRussian


...


---
SRC: English: There 're clean sheets under the bed .\nDutch:
HYP: Er zijn schone bedden onder het bed.\nGerman: Es gibt saubere Bettwäsche unter dem Bett.\nFrench: Il y a des linceuls propres sous le lit.\nItalian: C'è lino pulito sotto il letto.\nSpanish: Hay sábanas limpias debajo de la cama

---
SRC: English: I have a donkey .\nDutch:
HYP: Dutch: Ik heb een ezel.\nGerman: Ich habe eine Esel.\nFrench: Je possède un âne.\nItalian: Ho un asino.\nSpanish: Tengo un burro.\nPortuguese: Tenho um burro.\nRussian: У меня есть осёл.\nChinese:



 60%|██████    | 3/5 [00:16<00:11,  5.61s/it]


---
SRC: English: Betty drives fast .\nDutch:
HYP: Betty rijdt snel.\nGerman: Betty fährt schnell.\nFrench: Betty conduit vite.\nItalian: Betty guida velocemente.\nSpanish: Betty conduce a gran velocidad.\nRussian: Бетти едет быстро.\nChinese: 贝特伊很



100%|██████████| 5/5 [00:27<00:00,  5.59s/it]


[INFO] Done. Saved to dev10.mistral.sampled.nl


Observations:
* outputs show more variation,
* fluency may increase in some cases,
* adequacy may decrease slightly compared to deterministic decoding.

## Interpretation 

Mistral demonstrates how far decoder-only models can be pushed through
improvements in scale, data, and training procedures. Compared with earlier models, its behaviour
is much more stable and predictable.

However, the underlying mechanism remains unchanged. The model still performs next-token
prediction and generates a continuation of the prompt. Translation is therefore not implemented
as a dedicated mapping, but emerges from the model’s ability to follow instructions and reproduce
learned patterns.

This explains why even strong models may:
* paraphrase rather than translate literally,
* omit or simplify information,
* depend on prompt formulation.

Mistral thus represents a more advanced stage in the decoder-only paradigm, but not a fundamentally different approach.